# AEZ-26 Phase 2C — Flash-Drought Detection v3

## First-threshold-crossing correction

This version supersedes the earlier Phase 2C notebooks.

The detector now evaluates a candidate onset at the **first pentad at or below the 20th percentile** after a >40th-percentile anchor.

If that first dry excursion fails the rapid-intensification or minimum-duration criterion, the excursion is rejected and scanning resumes after its recovery. The previous anchor is **not** reused to connect a later dry excursion across an intervening recovery.

Primary definition:
- GLDAS `SM_0_100cm_mm`
- pentad scale
- >40th-percentile pre-onset anchor
- first subsequent <=20th-percentile crossing
- average decline >=5 percentile points/pentad
- minimum duration >=3 pentads
- recovery at >=20th percentile

Outputs are saved automatically to `Phase2/Phase2C_v3/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path('/content/drive/MyDrive/AEZ26_Final_Research_Data')
PHASE2_DIR = BASE_DIR / 'Phase2'
PHASE2A_DIR = PHASE2_DIR / 'Phase2A'
PHASE2C_DIR = PHASE2_DIR / 'Phase2C_v3'

PHASE2C_DIR.mkdir(parents=True, exist_ok=True)

# Preferred new Phase-2A location, with fallback to the earlier location.
input_candidates = [
    PHASE2A_DIR / 'AEZ26_Daily_Hydroclimate_Backbone_2000_2025.csv',
    PHASE2_DIR / 'AEZ26_Daily_Hydroclimate_Backbone_2000_2025.csv',
]

INPUT = next((p for p in input_candidates if p.exists()), None)

if INPUT is None:
    raise FileNotFoundError(
        'Daily backbone not found in Phase2/Phase2A or Phase2.'
    )

print('Using backbone:', INPUT)

daily = pd.read_csv(INPUT, parse_dates=['date']).sort_values('date').reset_index(drop=True)

expected_dates = pd.date_range('2000-01-01', '2025-12-31', freq='D')

assert len(daily) == 9497
assert daily['date'].is_unique
assert pd.DatetimeIndex(daily['date']).equals(expected_dates)
assert daily.isna().sum().sum() == 0

print('Input gate: PASS', daily.shape)


## 1. Climatologically aligned pentads

A 365-day seasonal calendar is used so leap years do not shift all seasonal pentad boundaries after February. February 29 is retained in the corresponding seasonal pentad.


In [ ]:
is_leap = daily['date'].dt.is_leap_year
doy = daily['date'].dt.dayofyear

after_feb29 = is_leap & (daily['date'].dt.month > 2)
clim_doy = doy - after_feb29.astype(int)

feb29 = (daily['date'].dt.month == 2) & (daily['date'].dt.day == 29)
clim_doy = clim_doy.where(~feb29, 59)

daily['clim_doy'] = clim_doy.astype(int)
daily['year'] = daily['date'].dt.year.astype(int)
daily['pentad_of_year'] = (((daily['clim_doy'] - 1) // 5) + 1).astype(int)

assert daily['pentad_of_year'].between(1, 73).all()
assert (daily.groupby('year')['pentad_of_year'].nunique() == 73).all()

print('73 climatological pentads/year: PASS')


In [ ]:
pentad = daily.groupby(['year', 'pentad_of_year'], as_index=False).agg(
    pentad_start=('date', 'min'),
    pentad_end=('date', 'max'),
    n_days=('date', 'count'),

    precipitation_sum_mm=('precipitation_mm_day', 'sum'),
    precipitation_mean_mm_day=('precipitation_mm_day', 'mean'),

    t2m_mean_C=('t2m_mean_C', 'mean'),
    t2m_max_C=('t2m_max_C', 'max'),
    t2m_min_C=('t2m_min_C', 'min'),
    dewpoint_mean_C=('dewpoint_mean_C', 'mean'),
    relative_humidity_mean_pct=('relative_humidity_mean_pct', 'mean'),
    VPD_mean_kPa=('VPD_mean_kPa', 'mean'),
    VPD_max_kPa=('VPD_max_kPa', 'max'),
    wind_speed_mean_m_s=('wind_speed_mean_m_s', 'mean'),

    solar_radiation_sum_MJ_m2=('solar_radiation_MJ_m2_day', 'sum'),
    solar_radiation_mean_MJ_m2_day=('solar_radiation_MJ_m2_day', 'mean'),
    net_solar_radiation_sum_MJ_m2=('net_solar_radiation_MJ_m2_day', 'sum'),
    net_thermal_radiation_sum_MJ_m2=('net_thermal_radiation_MJ_m2_day', 'sum'),
    net_radiation_sum_MJ_m2=('net_radiation_MJ_m2_day', 'sum'),
    net_radiation_mean_MJ_m2_day=('net_radiation_MJ_m2_day', 'mean'),

    total_evaporation_sum_mm=('total_evaporation_mm_day', 'sum'),
    total_evaporation_mean_mm_day=('total_evaporation_mm_day', 'mean'),
    ERA5_potential_evaporation_sum_mm=('ERA5_potential_evaporation_mm_day', 'sum'),
    ERA5_potential_evaporation_mean_mm_day=('ERA5_potential_evaporation_mm_day', 'mean'),

    SM_0_10cm_mean_mm=('SM_0_10cm_mm', 'mean'),
    SM_10_40cm_mean_mm=('SM_10_40cm_mm', 'mean'),
    SM_40_100cm_mean_mm=('SM_40_100cm_mm', 'mean'),
    SM_100_200cm_mean_mm=('SM_100_200cm_mm', 'mean'),
    SM_0_40cm_mean_mm=('SM_0_40cm_mm', 'mean'),
    SM_0_100cm_mean_mm=('SM_0_100cm_mm', 'mean'),
    SM_0_200cm_mean_mm=('SM_0_200cm_mm', 'mean'),
    RootMoist_mean_mm=('RootMoist_mm', 'mean'),

    surface_runoff_sum_mm=('surface_runoff_mm_day', 'sum'),
    surface_runoff_mean_mm_day=('surface_runoff_mm_day', 'mean'),
    baseflow_sum_mm=('baseflow_subsurface_mm_day', 'sum'),
    baseflow_mean_mm_day=('baseflow_subsurface_mm_day', 'mean'),
    total_runoff_sum_mm=('total_runoff_mm_day', 'sum'),
    total_runoff_mean_mm_day=('total_runoff_mm_day', 'mean'),
    GLDAS_ET_sum_mm=('GLDAS_ET_mm_day', 'sum'),
    GLDAS_ET_mean_mm_day=('GLDAS_ET_mm_day', 'mean'),

    CHIRPS_valid_fraction_min=('CHIRPS_valid_pixel_fraction', 'min'),
    ERA5_complete_fraction=('ERA5_complete_day_flag', 'mean'),
    GLDAS_state_complete_fraction=('GLDAS_state_complete_flag', 'mean'),
    GLDAS_flux_complete_fraction=('GLDAS_flux_complete_flag', 'mean'),
    GLDAS_complete_fraction=('GLDAS_complete_day_flag', 'mean'),
)

pentad = pentad.sort_values('pentad_start').reset_index(drop=True)
pentad.insert(0, 'pentad_id', np.arange(1, len(pentad) + 1))

pentad['pentad_label'] = (
    pentad['year'].astype(str)
    + '_P'
    + pentad['pentad_of_year'].astype(str).str.zfill(2)
)

pentad['pentad_mid_date'] = (
    pentad['pentad_start']
    + (pentad['pentad_end'] - pentad['pentad_start']) / 2
)

pentad['leap_day_included_flag'] = (pentad['n_days'] == 6).astype(int)

assert len(pentad) == 1898
assert int((pentad['n_days'] == 6).sum()) == 7

print('Pentad rows:', len(pentad))
print(pentad['n_days'].value_counts().sort_index())


## 2. Seasonal soil-moisture percentiles

Primary = same climatological pentad across 2000–2025.

Sensitivity:
- `RootMoist_pct`;
- ±2-pentad seasonal-window percentile.


In [ ]:
pentad['SM_0_100_pct'] = (
    pentad.groupby('pentad_of_year')['SM_0_100cm_mean_mm']
    .rank(method='average', pct=True)
    .mul(100.0)
)

pentad['RootMoist_pct'] = (
    pentad.groupby('pentad_of_year')['RootMoist_mean_mm']
    .rank(method='average', pct=True)
    .mul(100.0)
)

def empirical_rank_percentile(value, reference):
    ref = np.asarray(reference, dtype=float)
    ref = ref[np.isfinite(ref)]

    if not np.isfinite(value) or len(ref) == 0:
        return np.nan

    less = np.sum(ref < value)
    equal = np.sum(ref == value)
    average_rank = less + (equal + 1.0) / 2.0

    return 100.0 * average_rank / len(ref)

window_pct = []

for row in pentad.itertuples(index=False):
    p = int(row.pentad_of_year)

    seasonal_window = [
        ((p - 1 + offset) % 73) + 1
        for offset in range(-2, 3)
    ]

    reference = pentad.loc[
        pentad['pentad_of_year'].isin(seasonal_window),
        'SM_0_100cm_mean_mm'
    ]

    window_pct.append(
        empirical_rank_percentile(
            row.SM_0_100cm_mean_mm,
            reference
        )
    )

pentad['SM_0_100_pct_window5'] = window_pct

pentad['SM_0_100_pct_change_1pentad'] = pentad['SM_0_100_pct'].diff()
pentad['SM_0_100_drying_rate_pp_1pentad'] = -pentad['SM_0_100_pct_change_1pentad']

print('Primary percentile missing:', pentad['SM_0_100_pct'].isna().sum())
print('RootMoist percentile missing:', pentad['RootMoist_pct'].isna().sum())
print('Window-5 percentile missing:', pentad['SM_0_100_pct_window5'].isna().sum())


## 3. Literature-consistent sequential event detector

The event detector uses the **first** <=20th-percentile crossing after a >40th-percentile anchor.

A short or slowly developing dry excursion that fails the event criteria is consumed through its recovery and rejected. The detector does not bridge that recovery to a later dry excursion using the old anchor.

This is the key v3 correction.


In [ ]:
def detect_flash_droughts(
    pentad_df,
    pct_col,
    catalogue_name,
    start_threshold=40.0,
    drought_threshold=20.0,
    recovery_threshold=20.0,
    min_decline_rate=5.0,
    min_event_duration_pentads=3,
    max_event_duration_pentads=None
):
    x = pentad_df.sort_values('pentad_start').reset_index(drop=True).copy()
    pct = x[pct_col].to_numpy(dtype=float)

    events = []
    i = 0
    event_number = 1
    n = len(x)

    while i < n - 1:

        # Candidate pre-onset anchor must be >40th percentile.
        if not np.isfinite(pct[i]) or pct[i] <= start_threshold:
            i += 1
            continue

        j = i + 1
        episode_handled = False

        while j < n:

            if not np.isfinite(pct[j]):
                i = j + 1
                episode_handled = True
                break

            # A new >40 pentad becomes the newer possible anchor.
            if pct[j] > start_threshold:
                i = j
                episode_handled = True
                break

            # IMPORTANT: evaluate the FIRST <=20 crossing only.
            if pct[j] <= drought_threshold:

                decline_rate = (pct[i] - pct[j]) / (j - i)

                # Find recovery for this first dry excursion.
                recovery_idx = j + 1

                while (
                    recovery_idx < n
                    and np.isfinite(pct[recovery_idx])
                    and pct[recovery_idx] < recovery_threshold
                ):
                    recovery_idx += 1

                right_censored = (
                    recovery_idx >= n
                    or (
                        recovery_idx < n
                        and not np.isfinite(pct[recovery_idx])
                    )
                )

                event_end_idx = (
                    n - 1
                    if right_censored
                    else recovery_idx - 1
                )

                onset_idx = i + 1
                duration = event_end_idx - onset_idx + 1

                qualifies = (
                    decline_rate >= min_decline_rate
                    and duration >= min_event_duration_pentads
                )

                if (
                    max_event_duration_pentads is not None
                    and duration > max_event_duration_pentads
                ):
                    qualifies = False

                if qualifies:

                    event_slice = x.loc[onset_idx:event_end_idx].copy()

                    severity = float(
                        np.maximum(
                            0.0,
                            start_threshold
                            - event_slice[pct_col].to_numpy(dtype=float)
                        ).sum()
                    )

                    ant_start = max(0, onset_idx - 4)
                    antecedent = x.loc[ant_start:onset_idx - 1].copy()

                    recovery_date = (
                        pd.NaT
                        if right_censored
                        else x.loc[recovery_idx, 'pentad_mid_date']
                    )

                    recovery_label = (
                        None
                        if right_censored
                        else x.loc[recovery_idx, 'pentad_label']
                    )

                    events.append({
                        'catalogue': catalogue_name,
                        'event_id': f'{catalogue_name}_FD_{event_number:03d}',

                        'anchor_pentad': x.loc[i, 'pentad_label'],
                        'anchor_mid_date': x.loc[i, 'pentad_mid_date'],
                        'anchor_percentile': float(pct[i]),

                        'onset_pentad': x.loc[onset_idx, 'pentad_label'],
                        'onset_start_date': x.loc[onset_idx, 'pentad_start'],
                        'onset_mid_date': x.loc[onset_idx, 'pentad_mid_date'],

                        'threshold_crossing_pentad': x.loc[j, 'pentad_label'],
                        'threshold_crossing_mid_date': x.loc[j, 'pentad_mid_date'],
                        'threshold_crossing_percentile': float(pct[j]),

                        'event_end_pentad': x.loc[event_end_idx, 'pentad_label'],
                        'event_end_date': x.loc[event_end_idx, 'pentad_end'],

                        'recovery_pentad': recovery_label,
                        'termination_mid_date': recovery_date,
                        'recovery_threshold_percentile': recovery_threshold,
                        'right_censored_flag': int(right_censored),

                        'intensification_duration_pentads': int(j - i),
                        'total_duration_pentads': int(duration),
                        'total_duration_actual_days': int(
                            event_slice['n_days'].sum()
                        ),

                        'mean_intensification_rate_pp_per_pentad': float(
                            decline_rate
                        ),
                        'minimum_soil_moisture_percentile': float(
                            event_slice[pct_col].min()
                        ),
                        'severity_40pct_deficit_pp_pentads': severity,

                        'prolonged_after_flash_flag': int(duration > 18),

                        'onset_year': int(x.loc[onset_idx, 'year']),
                        'onset_month': int(
                            x.loc[onset_idx, 'pentad_mid_date'].month
                        ),

                        # Descriptive forcing diagnostics only.
                        'antecedent_4pentad_precipitation_mm': float(
                            antecedent['precipitation_sum_mm'].sum()
                        ),
                        'antecedent_4pentad_VPD_mean_kPa': (
                            float(
                                np.average(
                                    antecedent['VPD_mean_kPa'],
                                    weights=antecedent['n_days']
                                )
                            )
                            if len(antecedent)
                            else np.nan
                        ),

                        'event_precipitation_mm': float(
                            event_slice['precipitation_sum_mm'].sum()
                        ),
                        'event_VPD_mean_kPa': float(
                            np.average(
                                event_slice['VPD_mean_kPa'],
                                weights=event_slice['n_days']
                            )
                        ),
                        'event_VPD_max_kPa': float(
                            event_slice['VPD_max_kPa'].max()
                        ),
                        'event_t2m_mean_C': float(
                            np.average(
                                event_slice['t2m_mean_C'],
                                weights=event_slice['n_days']
                            )
                        ),
                        'event_t2m_max_C': float(
                            event_slice['t2m_max_C'].max()
                        ),
                        'event_ERA5_potential_evaporation_mm': float(
                            event_slice[
                                'ERA5_potential_evaporation_sum_mm'
                            ].sum()
                        ),
                        'event_GLDAS_ET_mm': float(
                            event_slice['GLDAS_ET_sum_mm'].sum()
                        ),
                    })

                    event_number += 1

                # Whether accepted or rejected, this first dry excursion
                # is finished at recovery. Never bridge across recovery
                # using the old anchor.
                i = n if right_censored else recovery_idx
                episode_handled = True
                break

            j += 1

        if not episode_handled:
            i += 1

    return pd.DataFrame(events)


In [ ]:
# Primary Yuan/South-Asia-style catalogue — first-crossing implementation
events_primary = detect_flash_droughts(
    pentad,
    pct_col='SM_0_100_pct',
    catalogue_name='PRIMARY_SM0_100',
    recovery_threshold=20.0
)

# Sensitivity A — RootMoist
events_root = detect_flash_droughts(
    pentad,
    pct_col='RootMoist_pct',
    catalogue_name='SENS_ROOTMOIST',
    recovery_threshold=20.0
)

# Sensitivity C — ±2-pentad percentile window
events_window = detect_flash_droughts(
    pentad,
    pct_col='SM_0_100_pct_window5',
    catalogue_name='SENS_WINDOW5',
    recovery_threshold=20.0
)

# Sensitivity B — India-style recovery / maximum duration
events_india = detect_flash_droughts(
    pentad,
    pct_col='SM_0_100_pct',
    catalogue_name='SENS_INDIA25_MAX18',
    recovery_threshold=25.0,
    max_event_duration_pentads=18
)

print('Primary events:', len(events_primary))
print('RootMoist sensitivity:', len(events_root))
print('Window-5 sensitivity:', len(events_window))
print('India-style recovery/max-duration sensitivity:', len(events_india))


In [ ]:
# Add primary event-phase flags to the pentad table.
pentad['primary_flash_drought_event_flag'] = 0
pentad['primary_flash_drought_event_id'] = ''
pentad['primary_flash_drought_intensification_flag'] = 0
pentad['primary_flash_drought_drought_phase_flag'] = 0

label_to_idx = {
    label: idx
    for idx, label in enumerate(pentad['pentad_label'])
}

for ev in events_primary.itertuples(index=False):

    onset_idx = label_to_idx[ev.onset_pentad]
    cross_idx = label_to_idx[ev.threshold_crossing_pentad]
    end_idx = label_to_idx[ev.event_end_pentad]

    pentad.loc[
        onset_idx:end_idx,
        'primary_flash_drought_event_flag'
    ] = 1

    pentad.loc[
        onset_idx:end_idx,
        'primary_flash_drought_event_id'
    ] = ev.event_id

    pentad.loc[
        onset_idx:cross_idx,
        'primary_flash_drought_intensification_flag'
    ] = 1

    pentad.loc[
        cross_idx:end_idx,
        'primary_flash_drought_drought_phase_flag'
    ] = 1

print(
    'Primary event pentads:',
    int(pentad['primary_flash_drought_event_flag'].sum())
)


## 4. Save all Phase 2C outputs automatically


In [ ]:
PENTAD_OUT = PHASE2C_DIR / 'AEZ26_Pentad_Hydroclimate_Indicators_2000_2025.csv'
PRIMARY_OUT = PHASE2C_DIR / 'AEZ26_Flash_Drought_Event_Catalogue_Primary_2000_2025.csv'
ROOT_OUT = PHASE2C_DIR / 'AEZ26_Flash_Drought_Event_Catalogue_Sensitivity_RootMoist_2000_2025.csv'
WINDOW_OUT = PHASE2C_DIR / 'AEZ26_Flash_Drought_Event_Catalogue_Sensitivity_Window5_2000_2025.csv'
INDIA_OUT = PHASE2C_DIR / 'AEZ26_Flash_Drought_Event_Catalogue_Sensitivity_India25_Max18_2000_2025.csv'
QC_OUT = PHASE2C_DIR / 'AEZ26_Phase2C_Flash_Drought_Detection_QC.csv'

pentad.to_csv(PENTAD_OUT, index=False, date_format='%Y-%m-%d')
events_primary.to_csv(PRIMARY_OUT, index=False, date_format='%Y-%m-%d')
events_root.to_csv(ROOT_OUT, index=False, date_format='%Y-%m-%d')
events_window.to_csv(WINDOW_OUT, index=False, date_format='%Y-%m-%d')
events_india.to_csv(INDIA_OUT, index=False, date_format='%Y-%m-%d')

qc_rows = [
    ['Daily input rows', len(daily), 'PASS'],
    ['Pentad rows', len(pentad), 'PASS'],
    ['5-day pentads', int((pentad['n_days'] == 5).sum()), 'PASS'],
    ['6-day leap pentads', int((pentad['n_days'] == 6).sum()), 'PASS'],
    ['Primary percentile missing', int(pentad['SM_0_100_pct'].isna().sum()), 'PASS'],
    ['RootMoist percentile missing', int(pentad['RootMoist_pct'].isna().sum()), 'PASS'],
    ['Window-5 percentile missing', int(pentad['SM_0_100_pct_window5'].isna().sum()), 'PASS'],
    ['Primary events', len(events_primary), 'RESULT'],
    ['RootMoist sensitivity events', len(events_root), 'RESULT'],
    ['Window-5 sensitivity events', len(events_window), 'RESULT'],
    ['India-style sensitivity events', len(events_india), 'RESULT'],
    [
        'Primary right-censored events',
        int(events_primary['right_censored_flag'].sum())
        if len(events_primary) else 0,
        'QC'
    ],
    [
        'Primary events with duration >18 pentads',
        int(events_primary['prolonged_after_flash_flag'].sum())
        if len(events_primary) else 0,
        'QC'
    ],
]

qc_df = pd.DataFrame(
    qc_rows,
    columns=['check', 'value', 'status']
)

qc_df.to_csv(QC_OUT, index=False)

print('SAVED AUTOMATICALLY IN:', PHASE2C_DIR)

for p in [
    PENTAD_OUT,
    PRIMARY_OUT,
    ROOT_OUT,
    WINDOW_OUT,
    INDIA_OUT,
    QC_OUT
]:
    print(' -', p)


In [ ]:
print('FINAL PHASE 2C v3 CHECK')
print('Pentad rows:', len(pentad))
print('5-day pentads:', int((pentad['n_days'] == 5).sum()))
print('6-day leap pentads:', int((pentad['n_days'] == 6).sum()))
print('Primary percentile missing:', int(pentad['SM_0_100_pct'].isna().sum()))
print('Primary events:', len(events_primary))
print('RootMoist sensitivity events:', len(events_root))
print('Window-5 sensitivity events:', len(events_window))
print('India-style sensitivity events:', len(events_india))

# ------------------------------------------------------------
# Methodological event audit
# ------------------------------------------------------------
label_to_idx = {
    label: idx
    for idx, label in enumerate(pentad['pentad_label'])
}

violations = {
    'anchor_not_above_40': 0,
    'crossing_above_20': 0,
    'rate_below_5': 0,
    'duration_below_3': 0,
    'crossing_not_first_le20': 0,
    'recovery_below_threshold': 0,
    'event_overlap': 0,
}

intervals = []

for ev in events_primary.itertuples(index=False):

    ai = label_to_idx[ev.anchor_pentad]
    oi = label_to_idx[ev.onset_pentad]
    ci = label_to_idx[ev.threshold_crossing_pentad]
    ei = label_to_idx[ev.event_end_pentad]
    ri = label_to_idx[ev.recovery_pentad] if isinstance(ev.recovery_pentad, str) else None

    violations['anchor_not_above_40'] += int(ev.anchor_percentile <= 40)
    violations['crossing_above_20'] += int(ev.threshold_crossing_percentile > 20)
    violations['rate_below_5'] += int(ev.mean_intensification_rate_pp_per_pentad < 5)
    violations['duration_below_3'] += int(ev.total_duration_pentads < 3)

    if ci > ai + 1:
        violations['crossing_not_first_le20'] += int(
            (pentad.loc[ai + 1:ci - 1, 'SM_0_100_pct'] <= 20).any()
        )

    if ri is not None:
        violations['recovery_below_threshold'] += int(
            pentad.loc[ri, 'SM_0_100_pct']
            < ev.recovery_threshold_percentile
        )

    intervals.append((oi, ei))

for a, b in zip(intervals[:-1], intervals[1:]):
    violations['event_overlap'] += int(b[0] <= a[1])

print()
print('METHOD VIOLATION COUNTS')
for k, v in violations.items():
    print(k, ':', v)

assert all(v == 0 for v in violations.values())

print()
print('Primary right-censored:',
      int(events_primary['right_censored_flag'].sum())
      if len(events_primary) else 0)

print('Primary prolonged-after-flash (>18 pentads):',
      int(events_primary['prolonged_after_flash_flag'].sum())
      if len(events_primary) else 0)

print()
print('First 10 primary events:')
display(events_primary.head(10))


## Next gate

This v3 notebook supersedes the prior Phase 2C outputs.

Do not tune thresholds to recover an earlier event count. The v3 event count is the result of the fixed first-threshold-crossing implementation.

Upload the v3 CSV outputs for independent audit before Phase 2D.
